# Phase C — Layer-Probe Analysis on the `per_group_vib` Checkpoint

**Goal:** characterize what each layer group of the trained `per_group_vib` model encodes. The ICTAI Section V.J discussion currently says "groups exist" without saying *what* they represent. This notebook fills that gap with two cheap probes per group.

**Method:**
1. Load the trained `per_group_vib` + WavLM-Base-Plus checkpoint (the 11.67% one).
2. Forward pass on ASVspoof 2019 LA *dev* set; collect `z_groups` (post-VIB compressed features, 256-dim per group) and labels (spoof, speaker_id).
3. For each group k, fit two sklearn LogisticRegression probes:
   - **bona-fide vs spoof** — sanity: every group should still carry it.
   - **speaker classification** — tells us how much speaker identity each group still encodes after compression + adversarial training.
4. Plot per-group accuracies in a grouped bar chart.

**No training; runs on T4 in ~30 minutes.**

Self-contained: Run All from scratch.


## 1. Environment Setup


In [ ]:
!pip install -q transformers soundfile librosa scikit-learn tqdm safetensors

from google.colab import drive
drive.mount('/content/drive')

import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} ({gpu.total_memory / 1e9:.1f} GB)')
else:
    raise RuntimeError('No GPU detected')


## 2. Copy GLADA Package to Local Runtime


In [ ]:
import shutil, os, sys

CANDIDATES = [
    "/content/drive/Othercomputers/Jakob's MacBook Air/Thesis/298-deliverables/glada-code/glada",
    '/content/drive/MyDrive/Jakob/glada_pkg/glada',
    '/content/drive/MyDrive/Jakob/glada_pkg',
    '/content/drive/MyDrive/Jakob/glada',
]
DRIVE_PKG = next((c for c in CANDIDATES if os.path.exists(os.path.join(c, '__init__.py'))), None)
if DRIVE_PKG is None:
    raise FileNotFoundError(
        'No glada/__init__.py found in candidates. Run:\n'
        "  !find /content/drive -name '__init__.py' -path '*/glada/*' 2>/dev/null | head\n"
        'and add the parent directory to CANDIDATES above.'
    )

LOCAL_PKG = '/content/glada'
if os.path.exists(LOCAL_PKG): shutil.rmtree(LOCAL_PKG)
shutil.copytree(DRIVE_PKG, LOCAL_PKG)

for mod in list(sys.modules):
    if mod == 'glada' or mod.startswith('glada.'): del sys.modules[mod]
sys.path = [p for p in sys.path if 'glada' not in p]
if '/content' not in sys.path: sys.path.insert(0, '/content')
print(f'Package source: {DRIVE_PKG}\nCopied to:      {LOCAL_PKG}')


## 3. Datasets (ASVspoof 2019 LA + ITW)


In [ ]:
import tarfile

def extract_dataset(tar_path, extract_to, name):
    if os.path.exists(extract_to) and len(os.listdir(extract_to)) > 0:
        print(f'{name}: already extracted to {extract_to}'); return
    os.makedirs(extract_to, exist_ok=True)
    print(f'{name}: extracting {tar_path} -> {extract_to} ...')
    with tarfile.open(tar_path) as t: t.extractall(extract_to)
    print(f'{name}: done.')

extract_dataset('/content/drive/MyDrive/Jakob/Datasets/LA.tar', '/content/data/asvspoof', 'ASVspoof 2019 LA')
extract_dataset('/content/drive/MyDrive/Jakob/Datasets/release_in_the_wild.tar', '/content/data/itw', 'In-the-Wild')

ASV_PROTO      = '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt'
ASV_AUDIO      = '/content/data/asvspoof/LA/ASVspoof2019_LA_train/flac'
ASV_EVAL_PROTO = '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt'
ASV_EVAL_AUDIO = '/content/data/asvspoof/LA/ASVspoof2019_LA_eval/flac'
ITW_META       = '/content/data/itw/release_in_the_wild/meta.csv'
ITW_AUDIO      = '/content/data/itw/release_in_the_wild'
for p in [ASV_PROTO, ASV_AUDIO, ASV_EVAL_PROTO, ITW_META]:
    assert os.path.exists(p), f'Missing: {p}'
print('Datasets verified.')


## 4. Imports


In [ ]:
from glada import GLADAConfig, BACKBONE_REGISTRY, set_seed
from glada.datasets import (
    ASVspoofDatasetWithSpeaker,
    ASVspoofEvalDataset,
    InTheWildDataset,
    create_dataloaders,
)
from glada.augmentation import RawBoost
from glada.models import GLADAModel
from glada.models.vib import VIBBottleneck
from glada.models.grl import GradientReversal
from glada.models.discriminators import GroupedLayerDomainDiscriminators
from glada.trainers import GLADATrainer

import copy, gc, json, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

print('Imports OK')


## 5. INLINE: `PerGroupVIBModel` (with leave-one-out + per-group beta flags)

Subclasses `GLADAModel`. Adds K `VIBBottleneck` modules (one per group), each compressing a `(B, 2D)` group ASP output to `(B, bottleneck_dim)`. The per-group discriminator then receives the compressed `z_k` rather than the raw `g_k`.

**New flags vs. the phase_vib_stagger version:**
- `vib_disabled_groups: tuple = ()` — for these group indices, `z_k = mu_k` (deterministic, no reparam sampling) and `kl_k = 0`. Used by `phase_a_leaveout` to ablate one group's compression at a time.
- `vib_betas: tuple = None` — per-group beta override. If provided, `kl_total = sum_k vib_betas[k] * kl_k` instead of the default uniform mean. Used by `phase_b_confirm` to apply scaled betas based on Phase A leave-one-out results.

Backwards compatibility: with no extra kwargs, behavior matches the original phase_vib_stagger inline class.


In [ ]:
class PerGroupVIBModel(GLADAModel):
    """GLADAModel with one VIBBottleneck per group, applied BEFORE the
    per-group discriminators. The hierarchical pool + global VIB are
    replaced by a simple linear projection over the concatenated
    per-group compressed representations.

    Args (in addition to GLADAConfig):
        vib_disabled_groups: tuple of group indices to disable VIB on
            (z_k = mu_k, kl_k = 0). Used for leave-one-out ablation.
        vib_betas: tuple of length K with per-group beta multipliers.
            None -> uniform mean (original behavior). Tuple -> sum of
            beta-weighted KLs.
    """

    def __init__(self, config, vib_disabled_groups=(), vib_betas=None):
        super().__init__(config)
        K = config.num_groups
        D = config.hidden_dim
        bd = config.bottleneck_dim
        self.group_vibs = nn.ModuleList([
            VIBBottleneck(2 * D, bd) for _ in range(K)
        ])
        self.group_compressed_proj = nn.Linear(K * bd, bd)
        if self.glada_disc is not None:
            self.glada_disc = GroupedLayerDomainDiscriminators(
                num_groups=K,
                input_dim=bd,
                hidden_dim=config.glada_disc_hidden,
            )

        # Phase A controls.
        for k in vib_disabled_groups:
            assert 0 <= k < K, f'vib_disabled_groups index {k} not in [0,{K})'
        self._vib_disabled = set(vib_disabled_groups)
        if vib_betas is not None:
            assert len(vib_betas) == K, f'vib_betas must have len {K}, got {len(vib_betas)}'
            self.register_buffer('_vib_betas',
                torch.tensor(vib_betas, dtype=torch.float32))
        else:
            self._vib_betas = None

    def forward(self, input_values, attention_mask=None):
        backbone_frozen = not any(p.requires_grad for p in self.backbone.parameters())
        if backbone_frozen:
            with torch.no_grad():
                outputs = self.backbone(input_values, attention_mask=attention_mask, output_hidden_states=True)
        else:
            outputs = self.backbone(input_values, attention_mask=attention_mask, output_hidden_states=True)
        hidden_states = list(outputs.hidden_states)

        if attention_mask is not None:
            input_lengths = attention_mask.sum(dim=1)
            output_lengths = ((input_lengths - 1) // 320) + 1
            T = hidden_states[0].shape[1]
            frame_mask = (torch.arange(T, device=attention_mask.device).unsqueeze(0) < output_lengths.unsqueeze(1))
        else:
            frame_mask = None

        hidden_states = self.layer_norms(hidden_states)
        _, cka_loss, group_outputs = self.pooling(hidden_states, frame_mask=frame_mask)
        if not self.config.use_cka_loss:
            cka_loss = torch.tensor(0.0, device=group_outputs[0].device)

        # Per-group VIB with leave-one-out + per-group beta support.
        z_groups = []
        kl_per_group = []
        for k, g_k in enumerate(group_outputs):
            if k in self._vib_disabled:
                # Disabled: deterministic z_k = mu_k, kl_k = 0.
                z_k, _ = self.group_vibs[k](g_k, training=False)
                kl_k = torch.zeros((), device=g_k.device)
            else:
                z_k, kl_k = self.group_vibs[k](g_k, training=self.training)
            z_groups.append(z_k)
            kl_per_group.append(kl_k)

        # Aggregate KL across groups.
        if self._vib_betas is not None:
            # Beta-weighted sum (per-group beta).
            kl_total = sum(self._vib_betas[k] * kl_per_group[k] for k in range(len(group_outputs)))
        else:
            # Uniform mean (original behavior; effective per-group beta = config.vib_beta / G).
            kl_total = sum(kl_per_group) / max(len(group_outputs), 1)

        z_concat = torch.cat(z_groups, dim=-1)
        z = self.group_compressed_proj(z_concat)

        spoof_logits = self.task_classifier(z)
        result = {
            'spoof_logits': spoof_logits,
            'kl_loss': kl_total,
            'cka_loss': cka_loss,
            'features': z,
        }

        if self.glada_disc is not None:
            result['glada_domain_logits'] = self.glada_disc(z_groups, self.domain_grl_alpha)
            result['group_outputs'] = z_groups

        if self.domain_disc is not None:
            z_domain = GradientReversal.apply(z, self.domain_grl_alpha)
            result['domain_logits'] = self.domain_disc(z_domain)

        if self.speaker_adversary is not None:
            z_speaker = GradientReversal.apply(z, self.speaker_grl_alpha)
            result['speaker_logits'] = self.speaker_adversary(z_speaker)

        return result

print('PerGroupVIBModel defined (with vib_disabled_groups + vib_betas).')


## 6. Locate Trained Checkpoint

Edit `CHECKPOINT_DIR` below to point at the run directory of the trained `per_group_vib` Base-Plus model (the one that produced 11.67% ITW EER). The notebook auto-discovers the best checkpoint by reading the JSON results in that directory.


In [ ]:
# Path to the per_group_vib + Base-Plus run that produced 11.67% ITW EER.
# Run dir is something like: /content/drive/MyDrive/Jakob/Experiments/vib_stagger/per_group_vib_<timestamp>/
CHECKPOINT_DIR = '/content/drive/MyDrive/Jakob/Experiments/vib_stagger'

# Auto-discover the per_group_vib subdirectory if there are multiple runs.
import glob
candidate_runs = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, 'per_group_vib*')))
candidate_runs = [r for r in candidate_runs if os.path.isdir(r)]
if not candidate_runs:
    raise FileNotFoundError(
        f'No per_group_vib* subdirectories under {CHECKPOINT_DIR}. '
        f'Edit CHECKPOINT_DIR to the parent of the trained run.'
    )
RUN_DIR = candidate_runs[-1]  # latest by sort order (timestamps)
print(f'Using run dir: {RUN_DIR}')

# Best checkpoint is typically saved as 'best.pt' or 'checkpoint_best.pt'.
ckpt_candidates = ['checkpoint_best.pt', 'best.pt', 'best_checkpoint.pt']
CKPT_PATH = None
for c in ckpt_candidates:
    p = os.path.join(RUN_DIR, c)
    if os.path.exists(p):
        CKPT_PATH = p; break
if CKPT_PATH is None:
    pts = sorted(glob.glob(os.path.join(RUN_DIR, '*.pt')))
    if not pts:
        raise FileNotFoundError(f'No .pt checkpoints in {RUN_DIR}.')
    CKPT_PATH = pts[-1]
print(f'Loading checkpoint: {CKPT_PATH}')


## 7. Load Model & Data


In [ ]:
# Build a fresh PerGroupVIBModel with the saved config, then load weights.
import json as _json
config_json = os.path.join(RUN_DIR, 'config.json')
if os.path.exists(config_json):
    with open(config_json) as f: saved_cfg = _json.load(f)
    cfg = GLADAConfig(**{k: v for k, v in saved_cfg.items() if k in GLADAConfig.__dataclass_fields__})
else:
    print('No config.json in run dir; using defaults.')
    cfg = GLADAConfig(backbone='wavlm-base-plus', batch_size=64, num_workers=0,
                      phase1_epochs=3, phase2_epochs=17)
set_seed(cfg.seed)

model = PerGroupVIBModel(cfg).cuda()
state = torch.load(CKPT_PATH, map_location='cuda')
if 'model_state_dict' in state: state = state['model_state_dict']
missing, unexpected = model.load_state_dict(state, strict=False)
print(f'Loaded checkpoint. Missing keys: {len(missing)}, unexpected: {len(unexpected)}')
model.eval()

# Use the ASVspoof 2019 LA *dev* split for probing (train would overfit, eval is
# what we report). Need labeled spoof + speaker_id, both available via
# ASVspoofDatasetWithSpeaker on the dev protocol.
DEV_PROTO = '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt'
DEV_AUDIO = '/content/data/asvspoof/LA/ASVspoof2019_LA_dev/flac'
assert os.path.exists(DEV_PROTO), DEV_PROTO

probe_ds = ASVspoofDatasetWithSpeaker(
    protocol_path=DEV_PROTO, audio_dir=DEV_AUDIO,
    config=cfg, augmenter=None,  # no augmentation for probing
)
probe_loader = torch.utils.data.DataLoader(
    probe_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0,
)
print(f'Probe set: {len(probe_ds)} utterances, {len(probe_ds.speaker_to_id)} speakers')


## 8. Extract Per-Group Features

Run forward, capture `result['group_outputs']` (which is `z_groups` in PerGroupVIBModel), and stack across the dev split.


In [ ]:
from tqdm.auto import tqdm

all_z_groups = None  # will become list of K arrays of shape (N, bd)
all_spoof_labels = []
all_speaker_labels = []

with torch.no_grad():
    for batch in tqdm(probe_loader, desc='extracting features'):
        x = batch['input_values'].cuda(non_blocking=True)
        am = batch.get('attention_mask')
        if am is not None: am = am.cuda(non_blocking=True)
        out = model(x, attention_mask=am)
        z_groups = out['group_outputs']  # list of K tensors, each (B, bd)
        if all_z_groups is None:
            all_z_groups = [[] for _ in z_groups]
        for k, z in enumerate(z_groups):
            all_z_groups[k].append(z.float().cpu().numpy())
        all_spoof_labels.append(batch['label'].numpy())
        all_speaker_labels.append(batch['speaker_id'].numpy())

import numpy as np
all_z_groups = [np.concatenate(z_list, axis=0) for z_list in all_z_groups]
all_spoof_labels = np.concatenate(all_spoof_labels)
all_speaker_labels = np.concatenate(all_speaker_labels)
K = len(all_z_groups)
print(f'Extracted features: K={K} groups, N={all_z_groups[0].shape[0]}, bd={all_z_groups[0].shape[1]}')
print(f'Labels: spoof={all_spoof_labels.shape}, speaker={all_speaker_labels.shape} '
      f'({len(np.unique(all_speaker_labels))} unique speakers)')


## 9. Per-Group Linear Probes

Train a `LogisticRegression` per group on (a) spoof labels and (b) speaker labels. 80/20 split, balanced class weight, max_iter=1000. Report accuracy per group + chance baseline.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score

spoof_results = {}
speaker_results = {}

for k in range(K):
    z = all_z_groups[k]
    # spoof probe (binary)
    z_tr, z_te, y_tr, y_te = train_test_split(z, all_spoof_labels, test_size=0.2, random_state=42, stratify=all_spoof_labels)
    clf = LogisticRegression(max_iter=1000, class_weight='balanced', n_jobs=-1)
    clf.fit(z_tr, y_tr)
    spoof_results[k] = balanced_accuracy_score(y_te, clf.predict(z_te))
    # speaker probe (multinomial)
    z_tr, z_te, y_tr, y_te = train_test_split(z, all_speaker_labels, test_size=0.2, random_state=42, stratify=all_speaker_labels)
    clf = LogisticRegression(max_iter=1000, multi_class='multinomial', class_weight='balanced', n_jobs=-1)
    clf.fit(z_tr, y_tr)
    speaker_results[k] = accuracy_score(y_te, clf.predict(z_te))
    print(f'  group {k}: spoof bal-acc={spoof_results[k]:.3f}, speaker acc={speaker_results[k]:.3f}')

n_speakers = len(np.unique(all_speaker_labels))
speaker_chance = 1.0 / n_speakers
print(f'\nSpeaker chance: {speaker_chance:.4f} ({n_speakers} speakers)')
print(f'Spoof chance:   0.500 (balanced binary)')


## 10. Plot

Two-row grouped bar chart: spoof accuracy and speaker accuracy per group.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
groups = list(range(K))

axes[0].bar(groups, [spoof_results[k] for k in groups], color='C0')
axes[0].axhline(0.5, ls='--', color='gray', label='chance')
axes[0].set_xticks(groups); axes[0].set_xlabel('Group index')
axes[0].set_ylabel('Spoof balanced accuracy')
axes[0].set_title('Spoof signal per group')
axes[0].legend()

axes[1].bar(groups, [speaker_results[k] for k in groups], color='C1')
axes[1].axhline(speaker_chance, ls='--', color='gray', label=f'chance ({speaker_chance:.3f})')
axes[1].set_xticks(groups); axes[1].set_xlabel('Group index')
axes[1].set_ylabel('Speaker accuracy')
axes[1].set_title('Speaker identity retained per group')
axes[1].legend()

plt.tight_layout()
fig_path = os.path.join(RUN_DIR, 'phase_c_layer_probe.png')
plt.savefig(fig_path, dpi=150)
print(f'Saved: {fig_path}')
plt.show()

# Save numerics to JSON for the paper.
out = {
    'checkpoint': CKPT_PATH,
    'n_speakers': int(n_speakers),
    'speaker_chance': float(speaker_chance),
    'per_group': {
        str(k): {
            'spoof_balanced_accuracy': float(spoof_results[k]),
            'speaker_accuracy': float(speaker_results[k]),
        } for k in groups
    },
}
json_path = os.path.join(RUN_DIR, 'phase_c_layer_probe.json')
with open(json_path, 'w') as f: _json.dump(out, f, indent=2)
print(f'Saved: {json_path}')


## 11. Interpretation Template (for ICTAI Section V.J)

Replace the bracketed values with the printed numbers above. The narrative anchors what the paper claims about the groups.

> The trained per_group_vib model retains spoof-discriminative signal across all three groups (balanced accuracy [SPOOF_G0 / SPOOF_G1 / SPOOF_G2] for groups 0/1/2 on the dev split), confirming that the per-group bottleneck does not destroy task signal at any depth. Speaker-identity probes show a [PATTERN: increasing / decreasing / flat] trend across groups (accuracy [SPK_G0 / SPK_G1 / SPK_G2] vs [CHANCE]), consistent with WavLM's known late-layer concentration of speaker information. Group K is the [LATE / EARLY / MIDDLE] group and carries the most [LINGUISTIC / ACOUSTIC] signal under our compression budget.
